# Finetuning Granite Speech (LoRA)


In [1]:
!pip install -U "transformers>=4.52" accelerate torch torchaudio torchvision jiwer pandas tqdm torchcodec peft bitsandbytes


Looking in indexes: https://nexus.iisys.de/repository/ki-awz-pypi-group/simple, https://pypi.org/simple
  Using cached transformers-5.23.0-py3-none-any.whl.metadata (31 kB)
  Using cached accelerate-1.16.0-py3-none-any.whl.metadata (19 kB)
  Using cached torch-2.15.0-cp314-cp314-manylinux_2_28_x86_64.whl.metadata (36 kB)
  Using cached torchaudio-2.11.1-py3-none-manylinux_2_28_x86_64.whl.metadata (7.5 kB)
  Using cached torchvision-0.29.2-py3-none-manylinux_2_28_x86_64.whl.metadata (5.7 kB)
  Using cached jiwer-4.0.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached pandas-3.1.1-cp314-cp314-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (16 kB)
  Using cached torchcodec-0.17.0-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (9.9 kB)
  Using cached peft-0.22.0-py3-none-any.whl.metadata (15 kB)
  Using cached bitsandbytes-0.50.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
  Using cached huggingface_hub-2.4.0-py3-none-any.whl.metadata (16 kB)
  Using

## Setup

In [2]:
import os
import torch
from pathlib import Path
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

MODEL_ID = "ibm-granite/granite-speech-5.0-2b"

MANIFEST_DIR = Path("asr_finetune")
OUTPUT_DIR = MANIFEST_DIR / "checkpoint"

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForSpeechSeq2Seq.from_pretrained(MODEL_ID, dtype=torch.bfloat16).to("cuda:0")
print(f"Model loaded on {model.device} with dtype {model.dtype}")

Loading weights:   0%|          | 0/954 [00:00<?, ?it/s]

Model loaded on cuda:0 with dtype torch.bfloat16


## Load the Training and Dev Manifests

In [3]:
import json

MANIFEST_TRAIN = MANIFEST_DIR / "manifest_train_new.jsonl"
MANIFEST_DEV = MANIFEST_DIR / "manifest_dev_new.jsonl"

def read_manifest(path):
    with open(path, encoding="UTF-8") as rf:
        return [json.loads(line) for line in rf]

train_data = read_manifest(MANIFEST_TRAIN)[:2000]
dev_data = read_manifest(MANIFEST_DEV)[:100]

print(len(train_data), "train samples,", len(dev_data), "dev samples")
print(train_data[0])

2000 train samples, 100 dev samples
{'audio': '/training-1/mls/mls_english_opus/train/audio/3488/5225/3488_5225_001583.opus', 'prompt': 'Transcribe the text, put named entities into brackets.', 'text': "it wasn't so terrible dark with the snow on the ground and i was so close to him i could have put an apple in his pocket [joe] explained who was the man asked [moxlow] i thought he looked like john north said [montgomery]"}


## Prompt Construction

In [4]:
import torchaudio

def load_audio(audio_path):
    waveform, sample_rate = torchaudio.load(audio_path)

    waveform = torchaudio.functional.resample(
        waveform, sample_rate, 16000
    )

    waveform = waveform.mean(dim=0)
    return waveform

def build_chat_prompt(user_prompt):
    chat = [
        {"role": "system", "content": "You are Granite, developed by IBM. You are a helpful AI assistant."},
        {"role": "user", "content": f"<|audio|>{user_prompt}"},
    ]
    return processor.tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)

DEFAULT_PROMPT = "Transcribe the text, put named entities into brackets."

def build_user_prompt(sample):
    return DEFAULT_PROMPT

## LoRA Configuration (PEFT)

In [5]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 4,259,840 || all params: 2,317,401,436 || trainable%: 0.1838


## Collate Function (Audio + Prompt + Target Text)

In [6]:
def collate_fn(batch):
    wavs = [load_audio(sample["audio"]) for sample in batch]
    prompts = [build_chat_prompt(build_user_prompt(sample)) for sample in batch]
    inputs = processor(prompts, wavs, return_tensors="pt", padding=True)

    eos_id = processor.tokenizer.eos_token_id   # the model must learn to stop
    target_ids = []
    for sample in batch:
        t = processor.tokenizer(sample["text"], add_special_tokens=False,
                                return_tensors="pt").input_ids[0]
        target_ids.append(torch.cat([t, torch.tensor([eos_id], dtype=t.dtype)]))
    batch_size, prompt_len = inputs["input_ids"].shape
    max_len = prompt_len + max(len(t) for t in target_ids)
    pad_id = processor.tokenizer.pad_token_id if processor.tokenizer.pad_token_id is not None else 0

    input_ids = torch.full((batch_size, max_len), pad_id, dtype=inputs["input_ids"].dtype)
    attention_mask = torch.zeros((batch_size, max_len), dtype=inputs["attention_mask"].dtype)
    labels = torch.full((batch_size, max_len), -100, dtype=torch.long)

    input_ids[:, :prompt_len] = inputs["input_ids"]
    attention_mask[:, :prompt_len] = inputs["attention_mask"]
    for i, t in enumerate(target_ids):
        input_ids[i, prompt_len:prompt_len + len(t)] = t
        attention_mask[i, prompt_len:prompt_len + len(t)] = 1
        labels[i, prompt_len:prompt_len + len(t)] = t

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        **{k: v for k, v in inputs.items() if k not in ("input_ids", "attention_mask")},
    }

batch = collate_fn(train_data[:2])
mask = batch["labels"] != -100
print(processor.tokenizer.decode(batch["labels"][0][mask[0]]))

it wasn't so terrible dark with the snow on the ground and i was so close to him i could have put an apple in his pocket [joe] explained who was the man asked [moxlow] i thought he looked like john north said [montgomery]<|end_of_text|>


## Training Configuration

In [7]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR.as_posix(),
    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,
    learning_rate=1e-5,
    num_train_epochs=2,
    eval_strategy="epoch",
    logging_steps=10,
    logging_first_step=True,
    save_strategy="epoch",
    bf16=True,
    optim="paged_adamw_8bit",
    report_to=[],
    remove_unused_columns=False,
    disable_tqdm=False,
)


## Evaluation (Dev WER)

In [8]:
import jiwer
from tqdm import tqdm
import re
import string
symbols = '!"#$%&\'()*+,-./:;<=>?@^_`{|}~'
def normalize(text):
    text = text.translate(str.maketrans("", "", symbols)).lower().strip()
    text = re.sub(r" +", " ", text).strip()
    return text
    
def compute_dev_wer():
    model.eval()
    wers = []
    for sample in tqdm(dev_data):
        wav = load_audio(sample["audio"])
        text_prompt = build_chat_prompt(build_user_prompt(sample))
        inputs = processor(text_prompt, wav, device=model.device, return_tensors="pt").to(model.device)
        with torch.inference_mode():
            out = model.generate(**inputs, max_new_tokens=400, do_sample=False, num_beams=1,
                                 use_cache=True)
        hyp = processor.tokenizer.decode(
            out[0, inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
        hyp = normalize(hyp)
        label = normalize(sample["text"])
        print("="*30)
        print(f"PRED:\t\t{hyp}")
        print(f"LABEL\t\t{label}")
        print("="*30)
        
        wers.append(jiwer.wer(label, hyp))
    model.train()
    return sum(wers) / len(wers)



In [ ]:
print("Dev WER before training:", compute_dev_wer())

## Train

In [9]:
from transformers import Trainer, TrainerCallback


class ConsoleLogCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            print(f"step {state.global_step}: {logs}")


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=dev_data,
    data_collator=collate_fn,
    callbacks=[ConsoleLogCallback()],
)

trainer.train()


Epoch,Training Loss,Validation Loss
1,3.104481,2.831243
2,2.161132,2.149616


step 1: {'loss': 4.752870082855225, 'grad_norm': 10.268470764160156, 'learning_rate': 1e-05, 'epoch': 0.008}
step 10: {'loss': 4.488111284044054, 'grad_norm': 6.717832565307617, 'learning_rate': 9.640000000000001e-06, 'epoch': 0.08}
step 20: {'loss': 4.932509231567383, 'grad_norm': 11.879193305969238, 'learning_rate': 9.240000000000001e-06, 'epoch': 0.16}
step 30: {'loss': 4.599738311767578, 'grad_norm': 6.100949287414551, 'learning_rate': 8.84e-06, 'epoch': 0.24}
step 40: {'loss': 3.844822311401367, 'grad_norm': 5.840126037597656, 'learning_rate': 8.44e-06, 'epoch': 0.32}
step 50: {'loss': 4.0672157287597654, 'grad_norm': 5.246493339538574, 'learning_rate': 8.040000000000001e-06, 'epoch': 0.4}
step 60: {'loss': 3.7648288726806642, 'grad_norm': 4.893845081329346, 'learning_rate': 7.640000000000001e-06, 'epoch': 0.48}
step 70: {'loss': 3.4087909698486327, 'grad_norm': 5.239016532897949, 'learning_rate': 7.24e-06, 'epoch': 0.56}
step 80: {'loss': 3.7580081939697267, 'grad_norm': 7.286921

TrainOutput(global_step=250, training_loss=2.9795585079193114, metrics={'train_runtime': 596.9768, 'train_samples_per_second': 6.7, 'train_steps_per_second': 0.419, 'total_flos': 1.4405167928151936e+16, 'train_loss': 2.9795585079193114, 'epoch': 2.0})

In [10]:
print("Dev WER AFTER training:", compute_dev_wer())

  1%|          | 1/100 [00:01<03:10,  1.92s/it]

PRED:		that unforgotten footstep an equally unforgotten scream flashed into mrs whitelaws mind with these words of her husbands some one shut up there yes there was the solution of the mystery that had puzzled and tormented her so long
LABEL		that unforgotten footstep and equally unforgotten scream flashed into mrs whitelaws mind with these words of her husbands some one shut up there yes that was the solution of the mystery that had puzzled and tormented her so long


  2%|▏         | 2/100 [00:03<03:09,  1.93s/it]

PRED:		mr farguss theory of life was that everybody was placed in life to fulfil a divine purpose and invested with the power to fulfil it no no its not fatalism mr fargus used to say not predestination
LABEL		mr farguss theory of life was that everybody was placed in life to fulfil a divine purpose and invested with the power to fulfil it no no its not fatalism mr [fargus] used to say not predestination


  3%|▎         | 3/100 [00:05<03:10,  1.97s/it]

PRED:		in the course of the following days the worthy sofra arranged a series of hunts for rameses setting out toward the east from atribis around the canals they shot birds with arrows some they snared in an immense net trap which took in a number of tens of them
LABEL		in the course of the following days the worthy [sofra] arranged a series of hunts for [rameses] setting out toward the east from atribis around the canals they shot birds with arrows some they snared in an immense net trap which took in a number of [tens] of them


... (62 outputs left out)

 35%|███▌      | 35/100 [00:48<01:26,  1.33s/it]

PRED:		here avarice first the keen desire of gain rules in each heart and works in every brain alike the veteran dames and virgins feel
LABEL		here avarice [first] the keen desire of gain rules in each heart and works in every brain alike the veteran dames and virgins feel


 35%|███▌      | 35/100 [00:50<01:33,  1.44s/it]


KeyboardInterrupt: 

## Write Out the Checkpoint (LoRA Merged into the Base Model)

In [ ]:
merged_model = model.merge_and_unload()   # merge LoRA weights into the base model
merged_model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)

print(f"Checkpoint written to {OUTPUT_DIR}")